> Notebook-friendly copy of `part-I/1.3-numpy-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"gsw": "gsw", "pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# Solutions

**ℹ️ Reference solutions**

Worked solutions for every exercise in [1.3-numpy-exercises.ipynb](1.3-numpy-exercises.ipynb), including the real Argo walkthrough. Exercise 9 needs the `gsw` library, which the exercise's own cell installs.

## Exercise 1: Create and inspect

Build the following 3×4 field of temperatures (°C) as a numpy array:

`5.2 4.8 6.1 3.9 1.3 0.5 -0.8 -1.2 -2.6 -3.1 -1.9 0.2`

Print its `shape`, `ndim`, and `dtype`, and its overall mean rounded to two decimals.
Then create, and print, an all-zeros array of the same shape and an array of the integers
0 to 11 reshaped to the same shape.

In [ ]:
import numpy as np

field_celsius = np.array([
    [ 5.2,  4.8,  6.1,  3.9],
    [ 1.3,  0.5, -0.8, -1.2],
    [-2.6, -3.1, -1.9,  0.2],
])
print(field_celsius.shape, field_celsius.ndim, field_celsius.dtype)
print(round(float(field_celsius.mean()), 2))

print(np.zeros((3, 4)))
print(np.arange(12).reshape(3, 4))

## Exercise 2: Index and slice

Given

```python
a = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
```

print the last row, the first column, and the 2×2 sub-block formed by the first two rows and the last two columns.

Finally, take a slice of the first row, change its first element to `99.0`, and print the original array. Then repeat using `.copy()`. Explain the difference in a comment.

In [ ]:
import numpy as np

In [ ]:
a = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
print(a[-1])
print(a[:, 0])
print(a[0:2, 2:4])

In [ ]:
row = a[0]          # a slice is a view, not a copy
row[0] = 99.0
print(a)            # the original changed too

In [ ]:
b = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
row_copy = b[0].copy()
row_copy[0] = 99.0
print(b)            # unaffected: .copy() breaks the shared memory

## Exercise 3: Masking and np.where

Given

```python
temp_celsius = np.array([[-1.0, 12.0,  3.0],
                         [ 8.0, -2.0, 15.0],
                         [ 4.0,  0.5, 11.0]])
```

count how many cells exceed 10 °C and print those values. Then build a clipped field in which every value below 0 °C is set to 0, using `np.where`.

In [ ]:
import numpy as np

temp_celsius = np.array([[-1.0, 12.0,  3.0],
                         [ 8.0, -2.0, 15.0],
                         [ 4.0,  0.5, 11.0]])
mask = temp_celsius > 10.0
print(int(mask.sum()))
print(temp_celsius[mask])

clipped = np.where(temp_celsius < 0.0, 0.0, temp_celsius)
print(clipped)

## Exercise 4: Broadcasting

Given the field below, add a per-column offset (length 4) to every row, and separately add a per-row offset (length 3) to every column.

```python
field = np.array([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
col_offset = np.array([0.0, 1.0, 2.0, 3.0])
row_offset = np.array([10.0, 20.0, 30.0])
```

In [ ]:
import numpy as np

field = np.array([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
col_offset = np.array([0.0, 1.0, 2.0, 3.0])
row_offset = np.array([10.0, 20.0, 30.0])

print(field + col_offset)            # (3,4) + (4,) broadcasts across rows
print(field + row_offset[:, None])   # (3,4) + (3,1) broadcasts across columns

## Exercise 5: Axis reductions

Using the same `field` as exercise 4, print the mean of each row, the maximum of each column, and the index of the column whose mean is largest.

Compute the overall mean twice — once as a method on the array, once with the numpy function — and confirm they agree.

In [ ]:
import numpy as np

field = np.array([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])

print(field.mean(axis=1))                 # mean of each row
print(field.max(axis=0))                  # max of each column
print(int(field.mean(axis=0).argmax()))   # column with the largest mean

print(field.mean(), np.mean(field))       # method and function agree

## Exercise 6: Missing data and interpolation

Given `series = np.array([1.0, 2.0, np.nan, 4.0, 5.0])`, fill the missing value by linear interpolation against the integer index, then print both the NaN-aware mean and the ordinary mean to show the difference.

In [ ]:
import numpy as np

series = np.array([1.0, 2.0, np.nan, 4.0, 5.0])
x = np.arange(series.size)
good = ~np.isnan(series)

filled = series.copy()
filled[~good] = np.interp(x[~good], x[good], series[good])
print(filled)

print(np.nanmean(series))   # skips the gap
print(np.mean(series))      # nan: propagates the gap

## Exercise 7: Avoid the dtype trap

Given the integer field `precip_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])`, compute the anomalies (each value minus the field mean) as a float array, without truncation. Confirm the result's dtype is floating point.

In [ ]:
import numpy as np

precip_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])
anomaly = precip_mm - precip_mm.mean()   # numpy promotes to float automatically
print(anomaly)
print(anomaly.dtype)

## Exercise 8: Reshape and stack

1. Create the integers 0 to 11 as a 1D array, then reshape it into a 3×4 array and print both
   shapes.
2. Reshape the same data into a 4×3 array. Print it and say in a comment why the numbers appear
   in a different arrangement.
3. Use `reshape(-1)` to flatten your 3×4 array back to 1D, and confirm the shape.
4. Stack the 3×4 array on top of a row of zeros with `np.vstack`, and print the resulting shape.

In [ ]:
import numpy as np

flat = np.arange(12)
grid = flat.reshape(3, 4)
print(flat.shape, grid.shape)          # (12,) (3, 4)

other = flat.reshape(4, 3)
print(other)
# the data is stored in one flat block and read row by row, so changing the shape
# changes where each row ends, not the order of the values

print(grid.reshape(-1).shape)          # (12,) — -1 infers the length

stacked = np.vstack([grid, np.zeros(4)])
print(stacked.shape)                   # (4, 4)

## Exercise 9: Ocean float profiles

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/argo-float-cycle.png" alt="A diagram of one Argo float cycle, numbered one to seven, from deployment through drift, dive, ascent and satellite transmission" width="100%">

<em>One cycle of an Argo float: deployment, descent to a drifting depth of about 1000 m, ten days of
drift, a dive to the profiling depth, then the ascent during which the measurements are taken, and
transmission to satellite at the surface before the next cycle begins. The panel on the right shows
what one ascent produces — the temperature and salinity profiles you are about to work with.
Illustration &copy; Thomas Haessig, from
<a href="https://www.euro-argo.eu/Outreach/Educational-material/Discover-Argo-floats-for-kids/How-do-the-Argo-floats-fulfill-their-mission-in-the-Ocean">Euro-Argo ERIC's educational material</a>.</em>


[Argo](https://argo.ucsd.edu/) floats are autonomous instruments that drift with ocean currents,
periodically diving and surfacing while recording temperature, salinity, and pressure. Each dive
produces one *profile*: a set of measurements at many depth *levels*, with a single latitude,
longitude, and date attached.

This exercise uses real Argo data. Steps 1 to 8 need nothing beyond this subchapter: loading
arrays, inspecting shapes, rebuilding an axis, vectorised arithmetic, reductions along an axis,
boolean masks, and NaN-aware statistics. Steps 9 to 11 then plot what you computed.

**ℹ️ Beyond this subchapter**

Steps 9 to 11 use `matplotlib`, which gets its own treatment in 1.4. You need four functions and
nothing else, and each is named where it is wanted:

- [`plt.plot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html) — a line, or
  one line per column when handed a 2D array
- [`plt.errorbar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.errorbar.html) — a
  line with error bars
- [`plt.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html) — points
- `plt.xlabel`, `plt.ylabel`, `plt.title` — each takes a string

An array of numbers with no picture attached is hard to sanity-check, which is why these three
steps sit here rather than waiting for 1.4: the profiles are the fastest way to see whether Steps 4
to 6 produced something physical.

In [ ]:
# Pre-supplied: download and unzip the Argo data files.
import pooch

files = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/argo_float_data.zip",
    known_hash="sha256:dd913ba3450241bcd5d5697187ea68d398bdd7fd563a04cc74386834749c25f5",
    processor=pooch.Unzip(),
    path=pooch.os_cache("mlees"),
)
for f in sorted(files):
    print(f)

**Step 1.** Load each array, taking the name from the filename.

In [ ]:
import numpy as np

paths = {f.split("/")[-1]: f for f in files}
print(sorted(paths))

T = np.load(paths["T.npy"])            # temperature, °C
S = np.load(paths["S.npy"])            # salinity, g/kg
P = np.load(paths["P.npy"])            # pressure, dbar
date = np.load(paths["date.npy"])      # one date per profile
lat = np.load(paths["lat.npy"])
lon = np.load(paths["lon.npy"])
level = np.load(paths["levels.npy"])   # depth level index

**Step 2.** The shapes tell you how the arrays fit together.

In [ ]:
for name, arr in (("T", T), ("S", S), ("P", P),
                  ("date", date), ("lat", lat), ("lon", lon), ("level", level)):
    print(f"{name:6s} {arr.shape}")

# T, S and P are (78, 75). Their first axis, of length 78, is the one they share with
# level; their second axis, of length 75, is the one they share with date, lat and lon.
# So axis 0 runs down through depth and axis 1 runs across profiles.

**Step 3.** `level` is a regular sequence, so it can be rebuilt two ways.

In [ ]:
print(level[0], level[-1], level.size)

level_arange = np.arange(level[0], level[-1] + 1)          # stop is excluded
level_linspace = np.linspace(level[0], level[-1], level.size)   # both ends included

print(np.allclose(level_arange, level))
print(np.allclose(level_linspace, level))

**Step 4.** The density anomaly, as one vectorised expression.

In [ ]:
!pip install gsw

In [ ]:
from gsw import CT_from_t

a = 7.718e-1
b = -8.44e-2
c = -4.561e-3

conservative_temp = CT_from_t(S, T, P)          # salinity, temperature, pressure
relative_density = a * S + b * conservative_temp + c * conservative_temp**2

print(relative_density.shape, T.shape)
print(np.allclose(relative_density.shape, T.shape))

**Step 5.** Reduce across profiles, which is axis 1, leaving one value per depth.

In [ ]:
mean_T = T.mean(axis=1)
mean_S = S.mean(axis=1)
mean_P = P.mean(axis=1)
mean_density = relative_density.mean(axis=1)

std_T = T.std(axis=1)
std_S = S.std(axis=1)
std_P = P.std(axis=1)
std_density = relative_density.std(axis=1)

print(mean_T.shape, level.shape)
print("first three mean temperatures:", mean_T[:3])

**Step 6.** Those means are full of `nan`, because an ordinary mean propagates missing values.

In [ ]:
n_missing = np.isnan(T).sum()
print("missing values in T:", n_missing, "of", T.size)
print(f"that is {100.0 * n_missing / T.size:.1f} % of the array")

mean_T = np.nanmean(T, axis=1)
mean_S = np.nanmean(S, axis=1)
mean_P = np.nanmean(P, axis=1)
mean_density = np.nanmean(relative_density, axis=1)

std_T = np.nanstd(T, axis=1)
std_S = np.nanstd(S, axis=1)
std_P = np.nanstd(P, axis=1)
std_density = np.nanstd(relative_density, axis=1)

print("any nan left in the mean temperature?", np.isnan(mean_T).any())

**Step 7.** Label each depth, then select depths by mask.

In [ ]:
labels = np.where(mean_T > 10.0, "warm", "cold")
print(labels[:10])
print(level[:10])

cold_mask = mean_T < 5.0
print("depth levels colder than 5 °C:", level[cold_mask])

**Step 8.** Stack the summary, save it, and check the round trip.

In [ ]:
summary = np.vstack([level, mean_T, mean_S, mean_density])
print(summary.shape)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)
np.save("_files/argo_summary.npy", summary)
reloaded = np.load("_files/argo_summary.npy")
print(np.allclose(summary, reloaded, equal_nan=True))

**Step 9.** One line per profile, variable against depth.

In [ ]:
import matplotlib.pyplot as plt

for values, label in [
    (T, "Temperature (in degC)"),
    (S, "Salinity (in g/kg)"),
    (P, "Pressure (in dbar)"),
    (relative_density, "Relative density (in kg/m3)"),
]:
    plt.figure()
    plt.plot(values, level)      # values is (level, profile): one line per column
    plt.xlabel(label)
    plt.ylabel("Depth level")
    plt.title(f"{label.split(' (')[0]} profiles at different locations")
    plt.show()

**Step 10.** The mean profile of each variable, with the spread as horizontal error bars.

In [ ]:
for mean_values, std_values, label in [
    (mean_T, std_T, "Temperature (in degC)"),
    (mean_S, std_S, "Salinity (in g/kg)"),
    (mean_P, std_P, "Pressure (in dbar)"),
    (mean_density, std_density, "Relative density (in kg/m3)"),
]:
    plt.figure()
    # the variable is on the x-axis, so the spread is a horizontal bar: xerr, not yerr
    plt.errorbar(mean_values, level, xerr=std_values)
    plt.xlabel(f"{label} including simple error bars")
    plt.ylabel("Depth level")
    plt.title(f"Mean {label.split(' (')[0].lower()} profile")
    plt.show()

# The bars are widest near the surface and narrow with depth: the upper ocean is where
# season, weather and location differ most between profiles, while the deep water the floats
# sample is close to the same everywhere in this set.

**Step 11.** Where the profiles were taken.

In [ ]:
plt.figure()
plt.scatter(lon, lat, s=20)
plt.xlabel("Longitude (in degrees east)", fontsize=12)
plt.ylabel("Latitude (in degrees north)", fontsize=12)
plt.title("Where each profile was taken", fontsize=12)
plt.show()

# One point per profile, not per level. The points fall in a compact cloud rather than
# spreading over an ocean basin: this is a small set of floats drifting in one region, which
# is also why the deep parts of the mean profiles above agree so closely.